# Sesión 03 – ETL escalable con Spark

**Tema:** Transformaciones con DataFrames, joins distribuidos, funciones ventana, validaciones básicas y salida en Parquet.

**Propósito práctico:** Construir un ETL que lea datos, los transforme, integre tablas, calcule métricas por ventana y genere una salida verificable.

## 1. Crear la SparkSession

Esta práctica se ejecuta en modo local dentro del contenedor. Se usa un número moderado de particiones para un caso de laboratorio.

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number, sum, col, when, count

spark = (
    SparkSession.builder
    .appName("sesion3-etl-spark")
    .master("local[*]")
    .config("spark.ui.port", "4040")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)

spark

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/30 00:35:32 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/03/30 00:35:33 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


## 2. Extract – Leer datasets

Para la sesión 3 se trabaja con dos datasets relacionados: `ventas.csv` y `clientes.csv`.

In [2]:
ventas = spark.read.csv("/opt/data/ventas.csv", header=True, inferSchema=True)
clientes = spark.read.csv("/opt/data/clientes.csv", header=True, inferSchema=True)

ventas.show()
clientes.show()

+---+----------+----------+-----+
| id|cliente_id|     fecha|monto|
+---+----------+----------+-----+
|  1|       101|2024-01-01|  100|
|  2|       101|2024-01-02|  200|
|  3|       102|2024-01-01|  150|
|  4|       103|2024-01-03|  300|
+---+----------+----------+-----+

+----------+------+
|cliente_id|nombre|
+----------+------+
|       101|  Juan|
|       102|   Ana|
|       103|  Luis|
+----------+------+



## 3. Explorar estructura

Antes de transformar, verificamos esquema y tipos de datos.

In [3]:
ventas.printSchema()
clientes.printSchema()

root
 |-- id: integer (nullable = true)
 |-- cliente_id: integer (nullable = true)
 |-- fecha: date (nullable = true)
 |-- monto: integer (nullable = true)

root
 |-- cliente_id: integer (nullable = true)
 |-- nombre: string (nullable = true)



## 4. Transform – Limpieza básica

Aplicamos limpieza inicial:
- eliminar nulos críticos
- remover duplicados en clientes

In [4]:
ventas_limpio = ventas.dropna(subset=["id", "cliente_id", "fecha", "monto"])
clientes_limpio = clientes.dropDuplicates(["cliente_id"])

ventas_limpio.show()
clientes_limpio.show()

+---+----------+----------+-----+
| id|cliente_id|     fecha|monto|
+---+----------+----------+-----+
|  1|       101|2024-01-01|  100|
|  2|       101|2024-01-02|  200|
|  3|       102|2024-01-01|  150|
|  4|       103|2024-01-03|  300|
+---+----------+----------+-----+

+----------+------+
|cliente_id|nombre|
+----------+------+
|       101|  Juan|
|       102|   Ana|
|       103|  Luis|
+----------+------+



## 5. Join distribuido

Integramos ventas con clientes mediante `cliente_id`. Este paso permite conectar información transaccional con atributos descriptivos.

In [5]:
df_join = ventas_limpio.join(clientes_limpio, "cliente_id", "inner")
df_join.show()

+----------+---+----------+-----+------+
|cliente_id| id|     fecha|monto|nombre|
+----------+---+----------+-----+------+
|       101|  2|2024-01-02|  200|  Juan|
|       101|  1|2024-01-01|  100|  Juan|
|       102|  3|2024-01-01|  150|   Ana|
|       103|  4|2024-01-03|  300|  Luis|
+----------+---+----------+-----+------+



## 6. Función ventana

Calculamos:
- `ranking`: orden de compra por cliente
- `total_acumulado`: monto acumulado por cliente

In [6]:
window_spec = Window.partitionBy("cliente_id").orderBy("fecha")

df_window = (
    df_join
    .withColumn("ranking", row_number().over(window_spec))
    .withColumn("total_acumulado", sum("monto").over(window_spec))
)

df_window.show()

+----------+---+----------+-----+------+-------+---------------+
|cliente_id| id|     fecha|monto|nombre|ranking|total_acumulado|
+----------+---+----------+-----+------+-------+---------------+
|       101|  1|2024-01-01|  100|  Juan|      1|            100|
|       101|  2|2024-01-02|  200|  Juan|      2|            300|
|       102|  3|2024-01-01|  150|   Ana|      1|            150|
|       103|  4|2024-01-03|  300|  Luis|      1|            300|
+----------+---+----------+-----+------+-------+---------------+



## 7. Validaciones básicas

Revisamos nulos por columna, cantidad de registros y posibles duplicados en el identificador de venta.

In [7]:
nulos = df_window.select([
    count(when(col(c).isNull(), c)).alias(c) for c in df_window.columns
])

nulos.show()

+----------+---+-----+-----+------+-------+---------------+
|cliente_id| id|fecha|monto|nombre|ranking|total_acumulado|
+----------+---+-----+-----+------+-------+---------------+
|         0|  0|    0|    0|     0|      0|              0|
+----------+---+-----+-----+------+-------+---------------+



In [8]:
print("Total de registros:", df_window.count())

df_window.groupBy("id").count().filter("count > 1").show()

Total de registros: 4
+---+-----+
| id|count|
+---+-----+
+---+-----+



## 8. Revisar decisiones iniciales de performance

Primero observamos el plan de ejecución del join y de la transformación con ventana.

In [9]:
df_window.explain(True)

== Parsed Logical Plan ==
'Project [unresolvedstarwithcolumns(total_acumulado, 'sum('monto) windowspecdefinition('cliente_id, 'fecha ASC NULLS FIRST, unspecifiedframe$()), None)]
+- Project [cliente_id#18, id#17, fecha#19, monto#20, nombre#39, ranking#140]
   +- Project [cliente_id#18, id#17, fecha#19, monto#20, nombre#39, ranking#140, ranking#140]
      +- Window [row_number() windowspecdefinition(cliente_id#18, fecha#19 ASC NULLS FIRST, specifiedwindowframe(RowFrame, unboundedpreceding$(), currentrow$())) AS ranking#140], [cliente_id#18], [fecha#19 ASC NULLS FIRST]
         +- Project [cliente_id#18, id#17, fecha#19, monto#20, nombre#39]
            +- Project [cliente_id#18, id#17, fecha#19, monto#20, nombre#39]
               +- Join Inner, (cliente_id#18 = cliente_id#38)
                  :- Filter atleastnnonnulls(4, id#17, cliente_id#18, fecha#19, monto#20)
                  :  +- Relation [id#17,cliente_id#18,fecha#19,monto#20] csv
                  +- Deduplicate [cliente_id#3

## 9. Load – Escribir salida en Parquet

La salida del ETL se guarda en formato Parquet dentro de `/opt/artifacts/output`.

In [10]:
df_window.write.mode("overwrite").parquet("/opt/artifacts/output")

## 10. Verificación de salida

Leemos nuevamente el resultado para comprobar que el ETL se generó correctamente.

In [11]:
salida = spark.read.parquet("/opt/artifacts/output")
salida.show()

+----------+---+----------+-----+------+-------+---------------+
|cliente_id| id|     fecha|monto|nombre|ranking|total_acumulado|
+----------+---+----------+-----+------+-------+---------------+
|       101|  1|2024-01-01|  100|  Juan|      1|            100|
|       101|  2|2024-01-02|  200|  Juan|      2|            300|
|       102|  3|2024-01-01|  150|   Ana|      1|            150|
|       103|  4|2024-01-03|  300|  Luis|      1|            300|
+----------+---+----------+-----+------+-------+---------------+



## 11. Reto de extensión

1. Filtrar solo ventas mayores a 150 antes del join.
2. Agregar una columna de categoría de monto: bajo, medio, alto.
3. Justificar si conviene pensar en broadcast cuando una tabla es muy pequeña.

In [12]:
df_reto = (
    ventas_limpio
    .filter(col("monto") > 150)
    .join(clientes_limpio, "cliente_id", "inner")
    .withColumn(
        "categoria_monto",
        when(col("monto") < 150, "bajo")
        .when(col("monto") <= 250, "medio")
        .otherwise("alto")
    )
)

df_reto.show()

+----------+---+----------+-----+------+---------------+
|cliente_id| id|     fecha|monto|nombre|categoria_monto|
+----------+---+----------+-----+------+---------------+
|       101|  2|2024-01-02|  200|  Juan|          medio|
|       103|  4|2024-01-03|  300|  Luis|           alto|
+----------+---+----------+-----+------+---------------+



## 12. Cierre breve

**Preguntas para reflexión:**
- ¿Qué diferencia hay entre una transformación simple y un join distribuido?
- ¿Qué aporta la función ventana frente a un `groupBy` tradicional?
- ¿Por qué Parquet es una mejor salida que CSV para analítica posterior?

**Entregable sugerido:** notebook funcional con ETL ejecutado, salida en Parquet y breve interpretación.